# ADLINK PCI-9812 Radar Data Sampler
**Driver:** ADLINK PCIS-DASK (Windows)  
**Hardware:** 4-channel simultaneous, 12-bit ADC, up to 20 MS/s  
**Voltage range:** ±1 V or ±5 V (programmable)

> **Requirements:** Install the ADLINK PCIS-DASK driver first so `PCIS-DASK.dll` is on the system PATH.

In [ ]:
import sys
import ctypes
import time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

if sys.platform != 'win32':
    raise EnvironmentError('This notebook must run on Windows (PCIS-DASK driver required)')

print('Python', sys.version)
print('ctypes', ctypes.__version__)
print('numpy ', np.__version__)

## Constants

In [ ]:
# Trigger modes
TRIG_SOFTWARE = 0
TRIG_POST     = 1
TRIG_PRE      = 2
TRIG_MIDDLE   = 3
TRIG_DELAY    = 4

# Trigger sources
TRIG_SRC_CH0  = 0
TRIG_SRC_CH1  = 1
TRIG_SRC_CH2  = 2
TRIG_SRC_CH3  = 3
TRIG_SRC_EXT  = 4   # External digital trigger

# Trigger slope
TRIG_SLOPE_POS = 0
TRIG_SLOPE_NEG = 1

# Clock sources
INT_CLK = 0   # Internal 40 MHz
SIN_CLK = 1   # External sine wave
SQR_CLK = 2   # External square wave

# Voltage ranges
VRANGE_1V = 1.0
VRANGE_5V = 5.0

# DMA completion flag
DMA_DONE    = 0
DMA_RUNNING = 1

# ADC
ADC_BITS     = 12
ADC_COUNTS   = 2 ** ADC_BITS   # 4096
ADC_MIDPOINT = ADC_COUNTS // 2 # 2048

MASTER_CLK_HZ = 40_000_000     # 40 MHz internal timebase

## Helper Functions

In [ ]:
def sample_rate_to_divisor(sample_rate_hz):
    """Convert desired sample rate to clock divisor. Rate = 40 MHz / (divisor+1)."""
    divisor = round(MASTER_CLK_HZ / sample_rate_hz) - 1
    divisor = max(1, min(divisor, 65535))
    actual = MASTER_CLK_HZ / (divisor + 1)
    if abs(actual - sample_rate_hz) / sample_rate_hz > 0.01:
        print(f'[warn] Requested {sample_rate_hz/1e6:.3f} MS/s, '
              f'closest available is {actual/1e6:.3f} MS/s (divisor={divisor})')
    return divisor


def raw_to_voltage(raw, vrange=VRANGE_5V):
    """Convert 12-bit unsigned ADC counts to signed voltage."""
    return ((raw.astype(np.float32) - ADC_MIDPOINT) / ADC_MIDPOINT) * vrange

## PCI-9812 Driver Class

In [ ]:
class PCI9812:
    """
    Wrapper for the ADLINK PCIS-DASK Windows DLL.
    Supports use as a context manager:  with PCI9812() as daq: ...
    """

    DLL_NAME = 'PCIS-DASK.dll'

    def __init__(self, card_no=0, vrange=VRANGE_5V):
        self.card_no  = card_no
        self.vrange   = vrange
        self._dll     = None
        self._dma_buf = None
        self._open    = False

    # ------------------------------------------------------------------ lifecycle

    def open(self):
        # Use windll.LoadLibrary — stdcall convention, works on all Windows Pythons
        try:
            self._dll = ctypes.windll.LoadLibrary(self.DLL_NAME)
        except OSError as exc:
            raise RuntimeError(
                f'Cannot load {self.DLL_NAME}.\n'
                'Ensure the ADLINK PCIS-DASK driver package is installed.'
            ) from exc

        self._set_prototypes()

        op_base    = ctypes.c_uint16(0)
        pt_base    = ctypes.c_uint16(0)
        irq_no     = ctypes.c_uint16(0)
        pci_master = ctypes.c_uint16(0)

        ret = self._dll.W_9812_Initial(
            self.card_no,
            ctypes.byref(op_base),
            ctypes.byref(pt_base),
            ctypes.byref(irq_no),
            ctypes.byref(pci_master),
        )
        self._check(ret, 'W_9812_Initial')
        self._open = True
        print(f'[PCI-9812] card {self.card_no} opened  '
              f'(base=0x{op_base.value:04X}, IRQ={irq_no.value}, master={pci_master.value})')

    def close(self):
        if self._dma_buf is not None:
            self._dll.W_9812_Free_DMA_Mem(ctypes.byref(self._dma_buf))
            self._dma_buf = None
        if self._open and self._dll is not None:
            self._dll.W_9812_Close(self.card_no)
            self._open = False
            print(f'[PCI-9812] card {self.card_no} closed')

    def __enter__(self):
        self.open()
        return self

    def __exit__(self, *_):
        self.close()

    # ------------------------------------------------------------------ config

    def set_clock(self, sample_rate_hz, clk_src=INT_CLK):
        divisor = sample_rate_to_divisor(sample_rate_hz)
        ret = self._dll.W_9812_Set_Clk_Rate(
            self.card_no, clk_src, 0, ctypes.c_uint16(divisor)
        )
        self._check(ret, 'W_9812_Set_Clk_Rate')
        actual = MASTER_CLK_HZ / (divisor + 1)
        print(f'[PCI-9812] sample rate → {actual/1e6:.3f} MS/s (divisor={divisor})')
        return actual

    def set_trigger(self, mode=TRIG_SOFTWARE, src=TRIG_SRC_EXT,
                    slope=TRIG_SLOPE_POS, level=0, post_samples=0):
        ret = self._dll.W_9812_Set_Trig(
            self.card_no, mode, src, slope,
            ctypes.c_uint16(level),
            ctypes.c_uint16(post_samples),
        )
        self._check(ret, 'W_9812_Set_Trig')

    # ------------------------------------------------------------------ acquire

    def acquire(self, channels, samples_per_channel,
                sample_rate_hz=10_000_000,
                trigger_mode=TRIG_SOFTWARE,
                timeout_s=5.0):
        """
        Acquire data.  Returns dict {channel_index: float32_voltage_array}.

        channels            – list of ints, e.g. [0,1,2,3]
        samples_per_channel – samples to collect per channel
        sample_rate_hz      – aggregate sample rate (max 20 MS/s)
        trigger_mode        – TRIG_SOFTWARE, TRIG_POST, etc.
        timeout_s           – DMA completion timeout
        """
        if not self._open:
            raise RuntimeError('Card not opened. Call open() or use as context manager.')

        n_ch          = len(channels)
        total_samples = samples_per_channel * n_ch

        ch_mask = 0
        for ch in channels:
            if ch not in range(4):
                raise ValueError(f'Invalid channel {ch}. Must be 0–3.')
            ch_mask |= (1 << ch)

        actual_rate = self.set_clock(sample_rate_hz)
        self.set_trigger(mode=trigger_mode, post_samples=samples_per_channel)

        # Allocate DMA buffer
        DMABuf  = ctypes.c_uint16 * total_samples
        dma_buf = DMABuf()
        buf_bytes = total_samples * ctypes.sizeof(ctypes.c_uint16)

        ret = self._dll.W_9812_Alloc_DMA_Mem(
            self.card_no, buf_bytes, ctypes.byref(dma_buf)
        )
        self._check(ret, 'W_9812_Alloc_DMA_Mem')
        self._dma_buf = dma_buf

        # Start DMA
        ret = self._dll.W_9812_AD_DMA_Start(
            self.card_no,
            ctypes.c_uint16(ch_mask),
            total_samples,
            ctypes.byref(dma_buf),
        )
        self._check(ret, 'W_9812_AD_DMA_Start')
        print(f'[PCI-9812] DMA started — {n_ch} ch × {samples_per_channel} samples '
              f'@ {actual_rate/1e6:.3f} MS/s')

        # Poll for completion
        status   = ctypes.c_uint16(DMA_RUNNING)
        deadline = time.monotonic() + timeout_s
        while True:
            self._check(
                self._dll.W_9812_AD_DMA_Status(self.card_no, ctypes.byref(status)),
                'W_9812_AD_DMA_Status'
            )
            if status.value == DMA_DONE:
                break
            if time.monotonic() > deadline:
                self._dll.W_9812_AD_DMA_Stop(self.card_no)
                raise TimeoutError(f'DMA timed out after {timeout_s}s')
            time.sleep(0.001)

        print('[PCI-9812] DMA complete')

        # De-interleave channels
        raw_all = np.frombuffer(dma_buf, dtype=np.uint16).copy()
        return {
            ch: raw_to_voltage(raw_all[idx::n_ch][:samples_per_channel], self.vrange)
            for idx, ch in enumerate(channels)
        }

    # ------------------------------------------------------------------ internal

    def _set_prototypes(self):
        dll  = self._dll
        U16  = ctypes.c_uint16
        PU16 = ctypes.POINTER(U16)
        INT  = ctypes.c_int

        dll.W_9812_Initial.argtypes      = [INT, PU16, PU16, PU16, PU16]
        dll.W_9812_Initial.restype       = INT
        dll.W_9812_Close.argtypes        = [INT]
        dll.W_9812_Close.restype         = INT
        dll.W_9812_Set_Clk_Rate.argtypes = [INT, INT, INT, U16]
        dll.W_9812_Set_Clk_Rate.restype  = INT
        dll.W_9812_Set_Trig.argtypes     = [INT, INT, INT, INT, U16, U16]
        dll.W_9812_Set_Trig.restype      = INT
        dll.W_9812_Alloc_DMA_Mem.argtypes = [INT, INT, ctypes.c_void_p]
        dll.W_9812_Alloc_DMA_Mem.restype  = INT
        dll.W_9812_Free_DMA_Mem.argtypes  = [ctypes.c_void_p]
        dll.W_9812_Free_DMA_Mem.restype   = INT
        dll.W_9812_AD_DMA_Start.argtypes  = [INT, U16, INT, ctypes.c_void_p]
        dll.W_9812_AD_DMA_Start.restype   = INT
        dll.W_9812_AD_DMA_Status.argtypes = [INT, PU16]
        dll.W_9812_AD_DMA_Status.restype  = INT
        dll.W_9812_AD_DMA_Stop.argtypes   = [INT]
        dll.W_9812_AD_DMA_Stop.restype    = INT

    @staticmethod
    def _check(ret, fname):
        if ret != 0:
            raise RuntimeError(f'{fname} returned error code {ret:#06x}')


print('PCI9812 class ready')

## Acquisition Parameters
Edit the values in this cell before running the acquisition.

In [ ]:
CARD_NO            = 0               # First installed PCI-9812 (0-based)
CHANNELS           = [0, 1, 2, 3]   # Any subset of [0, 1, 2, 3]
SAMPLES_PER_CH     = 8192            # Samples per channel
                                     #   ≤ 32768 single ch, ≤ 16384 two ch, ≤ 8192 four ch @ 20 MS/s
SAMPLE_RATE_HZ     = 10_000_000      # 10 MS/s  (max 20 MS/s)
VOLTAGE_RANGE      = VRANGE_5V       # VRANGE_5V (±5V) or VRANGE_1V (±1V)
TRIGGER_MODE       = TRIG_SOFTWARE   # TRIG_SOFTWARE / TRIG_POST / TRIG_PRE / TRIG_MIDDLE
TIMEOUT_S          = 5.0             # Seconds before acquisition gives up
SAVE_FILE          = 'radar_data.npz'

print('Parameters set.')

## Run Acquisition

In [ ]:
with PCI9812(card_no=CARD_NO, vrange=VOLTAGE_RANGE) as daq:
    data = daq.acquire(
        channels=CHANNELS,
        samples_per_channel=SAMPLES_PER_CH,
        sample_rate_hz=SAMPLE_RATE_HZ,
        trigger_mode=TRIGGER_MODE,
        timeout_s=TIMEOUT_S,
    )

print('\nChannel statistics:')
for ch, v in sorted(data.items()):
    print(f'  CH{ch}: min={v.min():.4f} V  max={v.max():.4f} V  '
          f'mean={v.mean():.4f} V  rms={np.sqrt(np.mean(v**2)):.4f} V')

## Save Data

In [ ]:
np.savez(
    SAVE_FILE,
    **{f'ch{ch}': v for ch, v in data.items()},
    sample_rate_hz=np.float64(SAMPLE_RATE_HZ),
)
print(f'Saved → {SAVE_FILE}')

# Reload example:
# d = np.load('radar_data.npz')
# ch0_volts = d['ch0']

## Plot — Time Domain

In [ ]:
n_ch = len(data)
fig, axes = plt.subplots(n_ch, 1, figsize=(14, 3 * n_ch), sharex=True)
if n_ch == 1:
    axes = [axes]

for ax, (ch, volts) in zip(axes, sorted(data.items())):
    t_us = np.arange(len(volts)) / SAMPLE_RATE_HZ * 1e6
    ax.plot(t_us, volts, linewidth=0.6)
    ax.set_ylabel(f'CH{ch} (V)')
    ax.set_ylim(-VOLTAGE_RANGE * 1.05, VOLTAGE_RANGE * 1.05)
    ax.axhline(0, color='gray', linewidth=0.4, linestyle='--')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (µs)')
fig.suptitle(f'PCI-9812  |  {SAMPLE_RATE_HZ/1e6:.0f} MS/s  |  {SAMPLES_PER_CH} samples/ch',
             fontsize=13)
plt.tight_layout()
plt.show()

## Plot — Frequency Spectrum (FFT)

In [ ]:
fig, axes = plt.subplots(n_ch, 1, figsize=(14, 3 * n_ch), sharex=True)
if n_ch == 1:
    axes = [axes]

for ax, (ch, volts) in zip(axes, sorted(data.items())):
    N    = len(volts)
    win  = np.hanning(N)
    spec = np.abs(np.fft.rfft(volts * win)) * 2 / N
    freq = np.fft.rfftfreq(N, d=1.0 / SAMPLE_RATE_HZ) / 1e6  # MHz
    ax.plot(freq, 20 * np.log10(spec + 1e-9), linewidth=0.6)
    ax.set_ylabel(f'CH{ch} (dBV)')
    ax.set_ylim(-100, 10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Frequency (MHz)')
fig.suptitle('Frequency Spectrum  |  Hanning window', fontsize=13)
plt.tight_layout()
plt.show()